In [18]:
from langgraph.graph import StateGraph ,START,END 

from langchain_groq import ChatGroq
from dotenv import load_dotenv 
load_dotenv() 
from typing import TypedDict,Annotated 
from langchain_core.messages import HumanMessage,BaseMessage,AIMessage

from langgraph.checkpoint.memory import MemorySaver  
from langgraph.types import interrupt, Command



In [19]:
llm=ChatGroq(model='Llama-3.3-70b-Versatile') 

In [20]:
# llm.invoke('hi') for checking the llm api 


In [21]:
from langgraph.graph.message import add_messages
#here BaseMessage is abstract it can be any kind of message human,system,ai,etc 
# add_messages is a reducer by which we append the messages 

class ChatState(TypedDict):

    messages: Annotated[list[BaseMessage], add_messages]

In [22]:
#lets make a obejct for the persistence memory 
checkpointer=MemorySaver() 

#lets make our graph 
graph=StateGraph(ChatState) 


In [23]:
# Fixed logic for the chatting 
def Chat_with_LLM(state: ChatState) -> ChatState: 
    decision = interrupt({
        "type": "approval",
        "reason": "Model is about to answer a user question.",
        "question": state["messages"][-1].content,
        "instruction": "Approve this question? yes/no"
    })
    
    if decision["approved"] == 'no':
        return {"messages": [AIMessage(content="Not approved.")]}

    else:
        response = llm.invoke(state["messages"])
        return {"messages": [response]}

In [24]:
# #logic for the chatting 
# def Chat_with_LLM(state: ChatState)-> ChatState: 
#     decision=interrupt('do you want to execute the query by llm?')
#     if decision['approved']=='no': 
#         response=AIMessage(content='Not Approved')
#         return {'messages':[response]}
#     else: 
#         response=llm.invoke(state['messages']) 
#         return {'messages':[response]}

In [25]:
#lets add node thier is only 1 node in it 
#by which user can communicate with the llm 
graph.add_node('Chat_Node',Chat_with_LLM) 




#lets add edges 
graph.add_edge(START,'Chat_Node') 
graph.add_edge('Chat_Node',END)




In [26]:
chat_bot=graph.compile(checkpointer=checkpointer) 


In [27]:
msg={'messages':[HumanMessage(content='what is aqi')]}
config={'configurable':{'thread_id':'23'}}
# final_output=chat_bot.invoke(msg) 



In [28]:
final_state=chat_bot.invoke(msg,config=config)

In [29]:
#let firstly extract the interrupt from the message 
#and then we invoke our remaining workflow using this interrupt input 

ipt=final_state['__interrupt__'][0]

In [30]:
ipt.value

{'type': 'approval',
 'reason': 'Model is about to answer a user question.',
 'question': 'what is aqi',
 'instruction': 'Approve this question? yes/no'}

In [31]:
user_input=input(f'backend message is {ipt.value} do you want to approve this yes/no?')

In [32]:
print(final_state)

{'messages': [HumanMessage(content='what is aqi', additional_kwargs={}, response_metadata={}, id='db8e15a0-d0b2-4a92-a119-fa595f86a2ed')], '__interrupt__': [Interrupt(value={'type': 'approval', 'reason': 'Model is about to answer a user question.', 'question': 'what is aqi', 'instruction': 'Approve this question? yes/no'}, id='c0e9aae69625c2318b4e9a2ba898aca2')]}


In [33]:
final_result = chat_bot.invoke(
    Command(resume={"approved": user_input}),
    config=config,
)


In [34]:
final_result['messages'][-1].content

"AQI stands for Air Quality Index. It's a measure used to indicate the level of air pollution in a given area. The AQI is calculated based on the concentrations of several major air pollutants, including:\n\n1. Particulate Matter (PM2.5 and PM10)\n2. Ozone (O3)\n3. Nitrogen Dioxide (NO2)\n4. Sulfur Dioxide (SO2)\n5. Carbon Monoxide (CO)\n\nThe AQI is usually reported as a numerical value, with higher values indicating poorer air quality. The index is often categorized into different levels, such as:\n\n* Good (AQI: 0-50): Air quality is satisfactory, and air pollution poses little or no risk.\n* Moderate (AQI: 51-100): Air quality is acceptable, but some pollutants may be present at levels that could pose a health risk for sensitive individuals.\n* Unhealthy for sensitive groups (AQI: 101-150): Air quality is unhealthy for people with pre-existing medical conditions, such as asthma.\n* Unhealthy (AQI: 151-200): Air quality is unhealthy for everyone, and may cause respiratory problems.\